In [6]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from shapely import make_valid

# Cache OSMnx requests so repeated runs don't keep re-downloading the same data.
ox.settings.use_cache = True

# The public Overpass server's /status endpoint has been returning an empty response, so OSMnx
# can't tell whether a query slot is free and pauses 60 s before EVERY request (it looks frozen).
# Skip that status check; if the server is busy it will answer with a 429/502 error instead.
ox.settings.overpass_rate_limit = False

In [7]:
# Geocode Ann Arbor and the University of Michigan for comparison -- neither perfectly
# matches North Campus, which is why we use a numeric bounding box below instead.
aa = ox.geocode_to_gdf('Ann Arbor, Michigan, USA')
print('Ann Arbor total_bounds:', aa.total_bounds)

um = ox.geocode_to_gdf('University of Michigan, Ann Arbor, Michigan, USA')
print('UMich total_bounds:', um.total_bounds)

# Roughly the bottom-left and top-right corners of UM North Campus.
# OSMnx expects bbox values in the order (left, bottom, right, top).
bbox = (-83.725706, 42.287305, -83.704518, 42.302508)
bbox

Ann Arbor total_bounds: [-83.799572   42.222668  -83.675807   42.3238941]
UMich total_bounds: [-83.7502236  42.242226  -83.6649743  42.3125148]


(-83.725706, 42.287305, -83.704518, 42.302508)

In [8]:
# Retrieve all features within the bounding box that are tagged as buildings.
building = ox.features_from_bbox(bbox, tags={'building': True})

# Print the count of each geometry type 
print(building.geom_type.value_counts())

# Fix any invalid geometries in the building GeoDataFrame and display the first few rows. make_valid() requires shapely 2.0 or later.
building['geometry'] = make_valid(building['geometry'])
display(building.head())

# reset row index to numbers and drop other columns; we only want to display limited information in the interactive map. 
building = building[['geometry']].reset_index(drop=True)
building.head()

Polygon         366
MultiPolygon      2
Name: count, dtype: int64


geometry  \
element  id                                                           
relation 3874939  POLYGON ((-83.72129 42.29726, -83.72117 42.297...   
         3874940  POLYGON ((-83.72088 42.29643, -83.72085 42.295...   
         3874941  POLYGON ((-83.72084 42.29492, -83.72111 42.294...   
         3877244  POLYGON ((-83.71756 42.29006, -83.71676 42.290...   
         3883610  POLYGON ((-83.71168 42.29468, -83.71168 42.294...   

                  addr:city addr:housenumber addr:postcode addr:state  \
element  id                                                             
relation 3874939  Ann Arbor             1760         48105         MI   
         3874940  Ann Arbor             1770         48105         MI   
         3874941  Ann Arbor             1780         48105         MI   
         3877244  Ann Arbor             2000         48109         MI   
         3883610  Ann Arbor             2455           NaN         MI   

                          addr:street amenity    building building:levels  \
element  id                                                                 
relation 3874939      Broadway Street     NaN  apartments               4   
         3874940      Broadway Street     NaN  apartments               4   
         3874941      Broadway Street     NaN  apartments               4   
         3877244  Bonisteel Boulevard     NaN  university               3   
         3883610       Hayward Street     NaN  university               3   

                 community_centre  ... drinking_water internet_access  \
element  id                        ...                                  
relation 3874939              NaN  ...            NaN             NaN   
         3874940              NaN  ...            NaN             NaN   
         3874941              NaN  ...            NaN             NaN   
         3877244              NaN  ...            NaN             NaN   
         3883610              NaN  ...            NaN             NaN   

                 internet_access:fee internet_access:ssid leisure membership  \
element  id                                                                    
relation 3874939                 NaN                  NaN     NaN        NaN   
         3874940                 NaN                  NaN     NaN        NaN   
         3874941                 NaN                  NaN     NaN        NaN   
         3877244                 NaN                  NaN     NaN        NaN   
         3883610                 NaN                  NaN     NaN        NaN   

                 sauna sport swimming_pool toilets:wheelchair  
element  id                                                    
relation 3874939   NaN   NaN           NaN                NaN  
         3874940   NaN   NaN           NaN                NaN  
         3874941   NaN   NaN           NaN                NaN  
         3877244   NaN   NaN           NaN                NaN  
         3883610   NaN   NaN           NaN                NaN  

[5 rows x 106 columns]

,geometry
0,"POLYGON ((-83.72129 42.29726, -83.72117 42.297..."
1,"POLYGON ((-83.72088 42.29643, -83.72085 42.295..."
2,"POLYGON ((-83.72084 42.29492, -83.72111 42.294..."
3,"POLYGON ((-83.71756 42.29006, -83.71676 42.290..."
4,"POLYGON ((-83.71168 42.29468, -83.71168 42.294..."


In [9]:
# retrieve all features within the bounding box that are tagged as highways. 
roads = ox.features_from_bbox(bbox, tags={'highway': True})

# print the count of each geometry type 
print(roads.geom_type.value_counts())

# only keep line geometries
roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])]

# Inspect the most common highway values in this area before trimming columns.
print(roads['highway'].value_counts().head(8))

# reset row index and keep the highway and geometry columns while dropping other columns; we only want to display limited information in the interactive map. 
roads = roads[['highway', 'geometry']].reset_index(drop=True)
roads.head()

LineString    1946
Point          342
Polygon         44
Name: count, dtype: int64
highway
footway         1271
service          358
steps             93
cycleway          63
unclassified      35
tertiary          33
path              32
residential       30
Name: count, dtype: int64


,highway,geometry
0,unclassified,"LINESTRING (-83.7127 42.30039, -83.71282 42.30..."
1,tertiary,"LINESTRING (-83.71297 42.29036, -83.71313 42.2..."
2,service,"LINESTRING (-83.72461 42.29701, -83.72525 42.2..."
3,service,"LINESTRING (-83.71025 42.29122, -83.71035 42.2..."
4,service,"LINESTRING (-83.7127 42.29956, -83.71278 42.29..."


In [10]:
# Define OSM tags to search for specific amenity types (food/drink, etc.)
tags = {
    'amenity': [
        'restaurant', 'bar', 'biergarten', 'fast_food',
        'food_court', 'cafe', 'pub', 'ice_cream',
    ]
}

# Retrieve all features within the bounding box that match the tags.
amenity = ox.features_from_bbox(bbox, tags=tags)

# Print the count of each geometry type
print(amenity.geom_type.value_counts())

# Convert non-point geometries (Polygon, MultiPolygon) to centroids before filtering columns.
amenity_points = amenity[amenity.geom_type == "Point"].copy()
amenity_polys = amenity[amenity.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
amenity_polys["geometry"] = amenity_polys.geometry.centroid
amenity_points = pd.concat([amenity_points, amenity_polys], ignore_index=True)  # ignore_index=True avoids duplicate or misaligned indices when combining

# Drop entries missing the 'name' column (so only named places remain)
amenity_points = amenity_points.dropna(subset=['name'])

# Keep only the relevant columns: name, amenity type, geometry; reset the index
amenity_point = amenity_points[['name', 'amenity', 'geometry']].reset_index(drop=True)

# Display a count for each amenity type present in the results
print(amenity_point['amenity'].value_counts())

# If you see a UserWarning about geographic CRS, ignore it. We only use centroids for mapping, which are unaffected by the CRS.

Point      17
Polygon     7
Name: count, dtype: int64
amenity
restaurant    11
fast_food      8
cafe           5
Name: count, dtype: int64


/var/folders/2y/tsx8tv1j2sd18z0wgd8fy3wc0000gn/T/ipykernel_50787/2658919970.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  amenity_polys["geometry"] = amenity_polys.geometry.centroid


In [11]:
# Fastest possible interactive preview: one GeoDataFrame, one line.
amenity_point.explore(color='blue', tiles='OpenStreetMap')

In [12]:
# Replace with your own key from carto.com/basemaps/apikey 
CARTO_API_KEY = 'cb1_4ibn_2_c02fff929f6657b876074436'

In [13]:
# CartoDB Positron: a light, low-contrast basemap that keeps colored overlays readable.
# The {z}/{x}/{y}/{r} placeholders are filled in by Leaflet itself at render time,
# not by Python, so they stay as plain text here -- only the API key is substituted.
tile_url = f'https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png?key={CARTO_API_KEY}'

# required attribution for using the map tiles
attr = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

# Build the basemap as its own TileLayer
base_tiles = folium.TileLayer(tiles=tile_url, attr=attr, name='CartoDB Positron', control=False)

# Buildings first: this call creates the map itself (m), styled as solid dark grey polygons with no border.
m = building.explore(
    tiles=base_tiles,
    color='darkgrey',
    style_kwds={'fillOpacity': 0.8, 'weight': 0},
    name='buildings',
)

# Roads: pass m= to add this layer onto the SAME map instead of creating a new one.
roads.explore(
    m=m,
    color='black',
    style_kwds={'weight': 1.5, 'opacity': 0.7},
    name='roads',
)

# Amenities: color automatically by the 'amenity' column (categorical=True gives each
# unique value its own color from the cmap), with an automatic legend, and a
# tooltip (hover) / popup (click) showing the name and type for each point.
amenity_point.explore(
    m=m,
    column='amenity',
    categorical=True,
    cmap=['orange', 'yellow', 'darkgreen'],
    marker_kwds={'radius': 5},
    tooltip=['name', 'amenity'],
    popup=['name', 'amenity'],
    legend=True,
    name='amenities',
)

# Let users toggle the buildings, roads, and amenities layers independently.
folium.LayerControl(collapsed=False).add_to(m)

m

In [14]:
html_file = 'lab9_map.html'
m.save(html_file)
print('Saved:', html_file)

Saved: lab9_map.html


## Practice Questions

### Q1: Retrieve a new OSM feature

Choose another OSM feature on North Campus -- bicycle parking, bus stops, parking lots, benches, or another amenity -- and retrieve it with `ox.features_from_bbox()`, following the same two-step process as above (the `bbox` from Step 1 still applies). Use the OSM Wiki or Taginfo to find the right key-value pair.

### Q2: Add it to the map

Add the new feature to the map above as another chained layer (besides amenities, roads, and building). Give it a color distinct from the amenities layer, and a tooltip or popup showing at least one useful attribute. Make sure it shows up as a new legend item and keep all other map aesthetics (e.g., legend style, basemap style, etc.) the same. 

### Q3: Export and host

Export your updated map as an HTML file, host it on GitHub Pages (or another static host), and submit the public URL for the quiz.

## Additional Resources

[OSMnx documentation](https://osmnx.readthedocs.io/), [Pyrosm documentation](https://pyrosm.readthedocs.io/) (an alternative for file-based OSM extracts), [GeoPandas documentation](https://geopandas.org/), [Folium User Guide](https://python-visualization.github.io/folium/latest/user_guide.html), [OSM Wiki Map Features](https://wiki.openstreetmap.org/wiki/Map_features), [Taginfo](https://taginfo.openstreetmap.org/), and [Leaflet Providers Preview](https://leaflet-extras.github.io/leaflet-providers/preview/).


In [15]:
# Q1: bicycle parking. OSM Wiki: amenity=bicycle_parking, with useful sub-tags
# capacity=* (number of bikes), covered=yes/no, and bicycle_parking=* (stands, wall_loops, rack, ...).
bike = ox.features_from_bbox(bbox, tags={'amenity': 'bicycle_parking'})

# Print the count of each geometry type
print(bike.geom_type.value_counts())

# Mapped either as single nodes or as areas (polygons); convert areas to centroids like the amenities above.
bike_points = bike[bike.geom_type == 'Point'].copy()
bike_polys = bike[bike.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
bike_polys['geometry'] = bike_polys.geometry.centroid
bike_points = pd.concat([bike_points, bike_polys], ignore_index=True)

# Most racks have no name, so keep the descriptive tags instead and label missing values for the tooltip.
bike_point = bike_points[['amenity', 'bicycle_parking', 'capacity', 'covered', 'geometry']].reset_index(drop=True)
bike_point[['bicycle_parking', 'capacity', 'covered']] = bike_point[['bicycle_parking', 'capacity', 'covered']].fillna('unknown')
bike_point['amenity'] = 'bicycle parking'

print(bike_point['bicycle_parking'].value_counts())
bike_point.head()

Point      127
Polygon      9
Name: count, dtype: int64
bicycle_parking
stands        126
loops           4
unknown         3
wall_loops      2
lockers         1
Name: count, dtype: int64


/var/folders/2y/tsx8tv1j2sd18z0wgd8fy3wc0000gn/T/ipykernel_50787/2828482299.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  bike_polys['geometry'] = bike_polys.geometry.centroid


,amenity,bicycle_parking,capacity,covered,geometry
0,bicycle parking,unknown,unknown,unknown,POINT (-83.7254 42.2975)
1,bicycle parking,unknown,unknown,unknown,POINT (-83.72537 42.29661)
2,bicycle parking,stands,16,no,POINT (-83.72131 42.29487)
3,bicycle parking,stands,unknown,yes,POINT (-83.72052 42.29362)
4,bicycle parking,stands,126,no,POINT (-83.71489 42.29232)


In [16]:
# Q2: rebuild the same map (same basemap, buildings, roads, amenities) and chain the bicycle parking layer on top.
m = building.explore(
    tiles=base_tiles,
    color='darkgrey',
    style_kwds={'fillOpacity': 0.8, 'weight': 0},
    name='buildings',
)

roads.explore(
    m=m,
    color='black',
    style_kwds={'weight': 1.5, 'opacity': 0.7},
    name='roads',
)

amenity_point.explore(
    m=m,
    column='amenity',
    categorical=True,
    cmap=['orange', 'yellow', 'darkgreen'],
    marker_kwds={'radius': 5},
    tooltip=['name', 'amenity'],
    popup=['name', 'amenity'],
    legend=True,
    name='amenities',
)

# Bicycle parking: same categorical legend style as amenities, in a color not used above.
bike_point.explore(
    m=m,
    column='amenity',
    categorical=True,
    cmap=['blue'],
    marker_kwds={'radius': 4},
    tooltip=['bicycle_parking', 'capacity', 'covered'],
    popup=['bicycle_parking', 'capacity', 'covered'],
    legend=True,
    name='bicycle parking',
)

folium.LayerControl(collapsed=False).add_to(m)

m

In [17]:
# Q3: GitHub Pages serves index.html at the repo's root URL, so save the map under both names.
for html_file in ['lab9_map.html', 'index.html']:
    m.save(html_file)
    print('Saved:', html_file)

Saved: lab9_map.html
Saved: index.html
